In [ ]:
!pip install transformers datasets seqeval conllu -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = "/content/drive/MyDrive/NLP_Project"
GLOVE_PATH  = f"{DRIVE_DIR}/glove.6B.100d.txt"
MODEL_SAVE  = f"{DRIVE_DIR}/bilstm_crf_ner.pt"
PARQUET_DIR = f"{DRIVE_DIR}/ontonotes_parquet"
LABEL_PATH = f"{DRIVE_DIR}/label.json"

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from datasets import load_dataset
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
from collections import defaultdict
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
!wget -q http://nlp.stanford.edu/data/glove.6B.zip -P {DRIVE_DIR}
!unzip -q {DRIVE_DIR}/glove.6B.zip -d {DRIVE_DIR}

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
dataset = load_dataset("parquet", data_files={
    "train":      f"{PARQUET_DIR}/train.parquet",
    "validation": f"{PARQUET_DIR}/validation.parquet",
    "test":       f"{PARQUET_DIR}/test.parquet",
})

print(dataset)
print("\nFirst sample:")
print(dataset["train"][0])

In [ ]:
import json

# Build word vocabulary from training set
word2idx = {"<PAD>": 0, "<UNK>": 1}
for sample in dataset["train"]:
    for token in sample["tokens"]:
        if token.lower() not in word2idx:
            word2idx[token.lower()] = len(word2idx)

# Tag mappings
with open(LABEL_PATH) as f:
    tag2idx = json.load(f)

idx2tag    = {idx: tag for tag, idx in tag2idx.items()}
label_list = [idx2tag[i] for i in range(len(tag2idx))]

print(f"Number of tags: {len(label_list)}")
print(f"Label list: {label_list}")

In [ ]:
def load_glove(glove_path, word2idx, embed_dim):
    print("Loading GloVe vectors...")
    embed_matrix = np.random.uniform(-0.1, 0.1, (len(word2idx), embed_dim))
    embed_matrix[0] = 0

    found = 0
    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            word  = parts[0]
            vec   = np.array(parts[1:], dtype=np.float32)
            if word in word2idx:
                embed_matrix[word2idx[word]] = vec
                found += 1

    print(f"GloVe vectors found: {found}/{len(word2idx)}")
    return torch.FloatTensor(embed_matrix)

EMBED_DIM    = 100
glove_matrix = load_glove(GLOVE_PATH, word2idx, EMBED_DIM)
print(f"Embedding matrix shape: {glove_matrix.shape}")

In [ ]:
class NERDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx):
        self.samples  = data
        self.word2idx = word2idx
        self.tag2idx  = tag2idx

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        tokens = [self.word2idx.get(t.lower(), 1) for t in sample["tokens"]]
        tags   = sample["tags"]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(tags, dtype=torch.long)

def collate_fn(batch):
    tokens, tags = zip(*batch)
    lengths      = torch.tensor([len(t) for t in tokens])
    tokens_pad   = pad_sequence(tokens, batch_first=True, padding_value=0)
    tags_pad     = pad_sequence(tags,   batch_first=True, padding_value=-1)
    return tokens_pad, tags_pad, lengths

train_dataset = NERDataset(dataset["train"],      word2idx, tag2idx)
val_dataset   = NERDataset(dataset["validation"], word2idx, tag2idx)
test_dataset  = NERDataset(dataset["test"],       word2idx, tag2idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

In [ ]:
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags    = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        self.start_trans = nn.Parameter(torch.randn(num_tags))
        self.end_trans   = nn.Parameter(torch.randn(num_tags))

    def forward(self, emissions, tags, mask):
        """Compute negative log likelihood loss."""
        return -self._compute_log_likelihood(emissions, tags, mask)

    def decode(self, emissions, mask):
        """Viterbi decoding."""
        return self._viterbi_decode(emissions, mask)

    def _compute_log_likelihood(self, emissions, tags, mask):
        batch_size, seq_len, num_tags = emissions.shape

        tags = tags.clone()
        tags[tags == -1] = 0

        score = self.start_trans[tags[:, 0]] + emissions[:, 0].gather(
            1, tags[:, 0].unsqueeze(1)
        ).squeeze(1)

        for i in range(1, seq_len):
            m           = mask[:, i].float()
            emit_score  = emissions[:, i].gather(1, tags[:, i].unsqueeze(1)).squeeze(1)
            trans_score = self.transitions[tags[:, i-1], tags[:, i]]
            score       = score + (emit_score + trans_score) * m

        last_tag_idx = mask.long().sum(1) - 1
        last_tags    = tags.gather(1, last_tag_idx.unsqueeze(1)).squeeze(1)
        score        = score + self.end_trans[last_tags]
        normalizer   = self._compute_normalizer(emissions, mask)
        return (score - normalizer).mean()

    def _compute_normalizer(self, emissions, mask):
        batch_size, seq_len, num_tags = emissions.shape
        score = self.start_trans + emissions[:, 0]
        for i in range(1, seq_len):
            m     = mask[:, i].unsqueeze(1).unsqueeze(2)
            score_ = score.unsqueeze(2) + self.transitions.unsqueeze(0) + emissions[:, i].unsqueeze(1)
            score  = torch.logsumexp(score_, dim=1)
            score  = torch.where(m.squeeze(2).bool(), score, score)
        score = score + self.end_trans
        return torch.logsumexp(score, dim=1)

    def _viterbi_decode(self, emissions, mask):
        batch_size, seq_len, num_tags = emissions.shape
        viterbi  = self.start_trans + emissions[:, 0]
        backptr  = []

        for i in range(1, seq_len):
            m        = mask[:, i].unsqueeze(1)
            v_t      = viterbi.unsqueeze(2) + self.transitions.unsqueeze(0)
            best_scores, best_tags = v_t.max(dim=1)
            viterbi  = best_scores + emissions[:, i]
            viterbi  = torch.where(m.bool(), viterbi, viterbi)
            backptr.append(best_tags)

        viterbi  += self.end_trans
        best_last = viterbi.argmax(dim=1)
        best_path = [best_last]

        for bp in reversed(backptr):
            best_last = bp.gather(1, best_last.unsqueeze(1)).squeeze(1)
            best_path.append(best_last)

        best_path.reverse()
        return torch.stack(best_path, dim=1)

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags, glove_matrix, dropout=0.5):
        super().__init__()

        # Embedding layer initialized with GloVe
        self.embedding        = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(glove_matrix)

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )

        self.dropout    = nn.Dropout(dropout)
        self.linear     = nn.Linear(hidden_dim * 2, num_tags)
        self.crf        = CRF(num_tags)

    def forward(self, tokens, tags, mask):
        # Embeddings
        embeds = self.dropout(self.embedding(tokens))

        # BiLSTM
        lengths      = mask.sum(dim=1).cpu()
        packed       = pack_padded_sequence(embeds, lengths, batch_first=True, enforce_sorted=False)
        lstm_out, _  = self.lstm(packed)
        lstm_out, _  = pad_packed_sequence(lstm_out, batch_first=True)
        lstm_out     = self.dropout(lstm_out)

        # Emission scores
        emissions = self.linear(lstm_out)

        # CRF loss
        loss = self.crf(emissions, tags, mask.float())
        return loss

    def predict(self, tokens, mask):
        embeds       = self.embedding(tokens)
        lengths      = mask.sum(dim=1).cpu()
        packed       = pack_padded_sequence(embeds, lengths, batch_first=True, enforce_sorted=False)
        lstm_out, _  = self.lstm(packed)
        lstm_out, _  = pad_packed_sequence(lstm_out, batch_first=True)
        emissions    = self.linear(lstm_out)
        return self.crf.decode(emissions, mask.float())

# Initialize model
HIDDEN_DIM = 256
NUM_TAGS   = len(tag2idx)

model = BiLSTMCRF(
    vocab_size=len(word2idx),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_tags=NUM_TAGS,
    glove_matrix=glove_matrix,
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for tokens, tags, lengths in tqdm(loader, desc="Training"):
        tokens = tokens.to(device)
        tags   = tags.to(device)
        mask = tokens != 0

        tags_input = tags.clone()
        tags_input[tags_input == -1] = 0

        optimizer.zero_grad()
        loss = model(tokens, tags_input, mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader, idx2tag):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for tokens, tags, lengths in tqdm(loader, desc="Evaluating"):
            tokens     = tokens.to(device)
            mask       = tokens != 0
            preds      = model.predict(tokens, mask)

            for i, length in enumerate(lengths):
                pred_tags  = [idx2tag[p.item()] for p in preds[i][:length]]
                true_tags  = [idx2tag[t.item()] for t in tags[i][:length] if t.item() != -1]
                # make sure lengths match
                min_len    = min(len(pred_tags), len(true_tags))
                all_preds.append(pred_tags[:min_len])
                all_labels.append(true_tags[:min_len])

    f1        = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall    = recall_score(all_labels, all_preds)
    return precision, recall, f1, all_preds, all_labels

In [ ]:
NUM_EPOCHS = 20
best_f1    = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)

    train_loss            = train_epoch(model, train_loader, optimizer)
    precision, recall, f1, _, _ = evaluate(model, val_loader, idx2tag)

    scheduler.step(1 - f1)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val P      : {precision:.4f}")
    print(f"Val R      : {recall:.4f}")
    print(f"Val F1     : {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), MODEL_SAVE)
        print(f"Best model saved (F1: {best_f1:.4f})")

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_SAVE))

precision, recall, f1, all_preds, all_labels = evaluate(model, test_loader, idx2tag)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"\nDetailed report:")
print(classification_report(all_labels, all_preds))